# External data

| Source | Contents | Size / time | Code |
|---|---|---|---|
| Open-Meteo archive | hourly weather (temperature, dew point, precipitation, snow, wind, cloud) | small | `taxi/external/weather.py` |
| IEM ASOS (Iowa State) | METAR reports → hourly weather-event flags | small | `taxi/external/metar.py` |
| OPDI (EUROCONTROL) | flight events (stand, apron, taxiway, runway) | a few GB | `taxi/external/opdi.py` |
| adsb.lol globe_history | ADS-B traces → ground / low points around the airports | ~3–4 GB per day; downloading takes hours | `taxi/external/adsb.py` |
| ansperformance.eu | monthly taxi-out additional time table (Excel; manual download) | small | `work/external/Taxi-Out_Additional_Time.xlsx` |

Files that are already downloaded are skipped. Set a flag to `True` and run the download cell. The ADS-B and OPDI roots are `config.ADSB_ROOT` and `config.OPDI_ROOT`.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
import config
import run
print("raw data folder:", config.RAW.name)

raw data folder: data


In [2]:
WEATHER = False
METAR = False
OPDI = False
ADSB = False
MONTHS = ["2026-02", "2026-06"]

## Current state

In [3]:
weather = config.EXTERNAL / "weather_hourly.parquet"
if weather.exists():
    h = pd.read_parquet(weather, columns=["ADEP_mvt", "MVT_YEAR", "MVT_MONTH"])
    print("weather:", len(h), "hours |", h.groupby(["MVT_YEAR"]).MVT_MONTH.agg(["min", "max"]).to_dict("index"))
for f in ("metar_hourly.parquet", "metar_hourly_extra.parquet"):
    p = config.EXTERNAL / f
    if p.exists():
        m = pd.read_parquet(p, columns=["hour"])
        print(f, ":", m.hour.min(), "-", m.hour.max())
pts = sorted((config.ADSB_ROOT / "points").glob("20*.parquet"))
print("ADS-B point days:", pd.Series([p.stem[:7] for p in pts]).value_counts().sort_index().to_dict())
flt = sorted((config.OPDI_ROOT / "filtered").glob("*.parquet"))
print("OPDI filtered files:", len(flt), "|", flt[0].name if flt else "", "...", flt[-1].name if flt else "")

weather: 138480 hours | {2025: {'min': 1, 'max': 12}, 2026: {'min': 1, 'max': 7}}
metar_hourly.parquet : 2025-01-01 00:00:00 - 2026-07-31 23:00:00
metar_hourly_extra.parquet : 2026-02-01 00:00:00 - 2026-06-30 23:00:00
ADS-B point days: {'2024-12': 1, '2025-01': 31, '2025-02': 28, '2025-06': 30, '2025-07': 31, '2025-08': 31, '2025-12': 31, '2026-01': 31, '2026-02': 28, '2026-05': 1, '2026-06': 30, '2026-07': 31}
OPDI filtered files: 34 | flight_events_20241226_20250105.parquet ... flight_events_20260729_20260808.parquet


## Download

In [4]:
from taxi.external import adsb as A, metar as M, opdi as O, weather as W
if WEATHER:
    W.download("2025-01-01", "2025-12-31")
    W.download("2025-12-31", "2026-07-31")
    W.hourly()
if METAR:
    months = [tuple(int(x) for x in a.split("-")) for a in MONTHS]
    M.download(months)
    M.hourly(months)
if OPDI:
    O.download(MONTHS)
    O.filter_events()
if ADSB:
    ds = A.days(["2026-01-31:2026-02-28", "2026-05-31:2026-06-30"])
    A.pipeline(ds, workers=6, delete=True, streams=2)
print("done")

done
